# Prepare Western District walking-access layers

This notebook downloads the official Lands Department Public Open Space points and LCSD Outdoor Fitness Equipment points from CSDI. It uses GeoPandas in Hong Kong 1980 Grid (EPSG:2326) for metric buffers and clipping, then writes WGS84 GeoJSON for the web map.

Both current CSDI source layers provide locations as **points**. They do not contain park or garden parcel boundaries, so the result locates public-space facilities and clips their location records; it does not claim a clipped park footprint.


In [ ]:
from pathlib import Path
import json
import requests
import geopandas as gpd
import pandas as pd

ROOT = Path('..').resolve()
SOURCE_DIR = ROOT / 'data' / 'source'
OUTPUT = ROOT / 'static' / 'data'
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

ORIGIN_LON_LAT = (114.1347709, 22.2855625)  # Hill Court, 28 Hill Road
METRIC_CRS = 'EPSG:2326'
WEB_CRS = 'EPSG:4326'
SEED_EXTENT_M = 4000  # candidates for a 2 km walking-time setting
GREEN_URL = 'https://portal.csdi.gov.hk/csdi-webpage/file-api?dataset_id=landsd_rcd_1637304969323_68704&format=geojson&layer_name=POS'
FITNESS_URL = 'https://portal.csdi.gov.hk/csdi-webpage/file-api?dataset_id=lcsd_rcd_1670389505454_91091&format=geojson&layer_name=geotagging'
GREEN_FILE = SOURCE_DIR / 'public-open-space.geojson'
FITNESS_FILE = SOURCE_DIR / 'outdoor-fitness-equipment.geojson'


def download_if_missing(url: str, path: Path) -> None:
    if path.exists():
        return
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    path.write_bytes(response.content)


download_if_missing(GREEN_URL, GREEN_FILE)
download_if_missing(FITNESS_URL, FITNESS_FILE)
print('Sources ready:', GREEN_FILE.name, FITNESS_FILE.name)


In [ ]:
def read_geojson(path: Path) -> gpd.GeoDataFrame:
    frame = gpd.read_file(path)
    # GeoJSON defaults to WGS84 (RFC 7946) when a legacy CRS member is absent.
    if frame.crs is None:
        frame = frame.set_crs(WEB_CRS)
    return frame


green = read_geojson(GREEN_FILE)
fitness = read_geojson(FITNESS_FILE)
print('Public open-space source:', len(green), green.crs, green.geom_type.value_counts().to_dict())
print('Outdoor-fitness source:', len(fitness), fitness.crs, fitness.geom_type.value_counts().to_dict())


## Build the destination seed layer

The 4 km buffer is only a candidate-data extent. It does not define walking reach. The live map requests walking routes and applies the selected time and slope thresholds to each destination.


In [ ]:
origin = gpd.GeoSeries(
    gpd.points_from_xy([ORIGIN_LON_LAT[0]], [ORIGIN_LON_LAT[1]]),
    crs=WEB_CRS,
).to_crs(METRIC_CRS).iloc[0]
seed_buffer = gpd.GeoDataFrame(geometry=[origin.buffer(SEED_EXTENT_M)], crs=METRIC_CRS)

green_m = green.to_crs(METRIC_CRS)
fitness_m = fitness.to_crs(METRIC_CRS)
green_seed = gpd.clip(green_m, seed_buffer)
fitness_seed = gpd.clip(fitness_m, seed_buffer)
print(f'{len(green_seed)} public-space points and {len(fitness_seed)} fitness points in the seed extent')


In [ ]:
green_seed = green_seed.copy()
green_seed['kind'] = 'green'
green_seed['name'] = green_seed['NAME_EN'].fillna('Public open space')
green_seed['name_zh'] = green_seed.get('NAME_TC')
green_seed['address'] = green_seed.get('ADDRESS_EN')
green_seed['source'] = 'Lands Department CSDI Public Open Space'
green_seed['source_url'] = GREEN_URL

gym_name = fitness_seed['Name_en'] if 'Name_en' in fitness_seed else pd.Series('Outdoor fitness point', index=fitness_seed.index)
fitness_seed = fitness_seed.copy()
fitness_seed['kind'] = 'gym'
fitness_seed['name'] = gym_name.fillna('Outdoor fitness point')
fitness_seed['name_zh'] = fitness_seed.get('Name_cn')
fitness_seed['address'] = fitness_seed.get('Address_en')
fitness_seed['source'] = 'LCSD CSDI Outdoor Fitness Equipment'
fitness_seed['source_url'] = FITNESS_URL

combined = gpd.GeoDataFrame(pd.concat([green_seed, fitness_seed], ignore_index=True), crs=METRIC_CRS).to_crs(WEB_CRS)
combined.to_file(OUTPUT / 'access-points.geojson', driver='GeoJSON', index=False)
print('Wrote', OUTPUT / 'access-points.geojson', 'features:', len(combined))


## Clip to the walking-time contour

To clip the candidate locations to actual walking reach, run the app with a configured Google Maps Routes and Elevation key, request `POST /api/isochrone`, and save the returned `feature` as `data/source/walking_isochrone.json`. The 24-direction contour uses Google walking durations and route elevation screening; it is still a sampled estimate, not a complete pedestrian-network isochrone.

The clip below keeps facility points inside that polygon. Because the source records are points, it cannot clip park parcels or prove step-free entrances.


In [ ]:
ISOCHRONE_FILE = SOURCE_DIR / 'walking_isochrone.json'
if not ISOCHRONE_FILE.exists():
    raise FileNotFoundError('Save the configured app’s /api/isochrone feature to ' + str(ISOCHRONE_FILE))
document = json.loads(ISOCHRONE_FILE.read_text())
feature = document.get('feature') or (document if document.get('type') == 'Feature' else None)
if feature is None or not feature.get('geometry'):
    raise ValueError('No walking contour found; inspect the API message and slope-safe sample count')
reach = gpd.GeoDataFrame.from_features([feature], crs=WEB_CRS).to_crs(METRIC_CRS)
accessible_points = gpd.clip(combined.to_crs(METRIC_CRS), reach)
accessible_points.to_crs(WEB_CRS).to_file(OUTPUT / 'reachable-access-points.geojson', driver='GeoJSON', index=False)
print('Clipped reachable source points:', len(accessible_points))


## Interpretation and limits

The live map uses these destination records as routing targets; an open-space point may not mark its public entrance. The 100 m–2 km slider represents a time budget at a 1 m/s reference pace. Google Routes supplies route durations when the backend key is configured; the public Pages preview uses Lands Department 3D Pedestrian Route times. The 3–20° control screens the steepest sampled absolute route grade.

The contour is based on sampled routes and can miss route alternatives, alleys, stairs, kerbs, crossing quality, surface condition, closures, and step-free entrances. The input Open Space and Outdoor Fitness Equipment datasets are updated periodically; preserve their retrieval dates and licenses alongside field verification before making product decisions.
